In [0]:
# ============================================================
# PAYCE — Gold Layer Configuration
# Notebook: 03_gold_fraud_mart
# ============================================================

CATALOG = "payce"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

print("Payce Config")

In [0]:
spark.table(f"{SILVER}.paysim").printSchema()

In [0]:
%sql
select * from payce.silver.paysim limit 5

In [0]:
# ============================================================
# FRAUD MART — dim_transaction_type
# One row per transaction type, with a surrogate key
# ============================================================

from pyspark.sql.functions import monotonically_increasing_id, col

# Get distinct transaction types from PaySim
dim_type = (
    spark.table(f"{SILVER}.paysim")
    .select("type")
    .distinct()
    .withColumnRenamed("type", "type_name")
    .withColumn("type_id", monotonically_increasing_id())
    .select("type_id", "type_name")   # reorder 
)

In [0]:
dim_type.show(5)

In [0]:
# Write the table to Gold
dim_type.write
.format("delta")
.mode("overwrite")
.saveAsTable(f"{GOLD}.dim_transaction_type")

In [0]:
# Print the output of the new table in GOLD dim_transaction_type
spark.table(f"{GOLD}.dim_transaction_type").show(5)

In [0]:
# ============================================================
# FRAUD MART — dim_date
# Derived from PaySim 'step' (1 step = 1 hour)
# ============================================================

from pyspark.sql.functions import col, expr

# Build date dimension from distinct types
dim_date = (
    spark.table(f"{SILVER}.paysim")
    .select("step")
    .distinct()
    .withColumnRenamed("step", "date_id")  # step itself is the unique key
    .withColumn("day", expr("cast(date_id / 24 as int) + 1"))  # which day (1 step = 1 hour)
    .withColumn("hour", expr("date_id % 24"))    # hour of day (0-23)
    .select("date_id", "day", "hour")
)

# Write to Gold
dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_date")

In [0]:
# Print the dim_date to double check
spark.table(f"{GOLD}.dim_date").orderBy("date_id").show(10)

In [0]:
# ============================================================
# FRAUD MART — fct_transactions
# Grain: one row per transaction
# Links to dim_transaction_type and dim_date
# ============================================================

from pyspark.sql.functions import col, monotonically_increasing_id

# Load Paysim Silver + 2 dimensions
paysim = spark.table(f"{SILVER}.paysim")
dim_type = spark.table(f"{GOLD}.dim_transaction_type")
dim_date = spark.table(f"{GOLD}.dim_date")

In [0]:
# Build the fact table
fct = (
    paysim.alias("p")
    
    # join to type dimension to get type_id
    .join(dim_type.alias("t"), col("p.type") == col("t.type_name"), "left")

    # join to date dimension to get date_id 
    .join(dim_date.alias("d"), col("p.step") == col("d.date_id"), "left")

    # Add a unique transaction key
    .withColumn("transaction_id", monotonically_increasing_id())
    
    # select fact columns: Keys + measures
    .select(
        "transaction_id",
        col("t.type_id"),
        col("d.date_id"),
        col("p.amount"),
        col("p.isFraud").alias("is_fraud"),
        col("p.isFlaggedFraud").alias("is_flagged_fraud")
    )
)

# Write to Gold
fct.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.fct_transactions")

In [0]:
# Check the columns
spark.table(f"{GOLD}.fct_transactions").show(5)

In [0]:
%sql
select count(*) from payce.gold.fct_transactions 

In [0]:
# ============================================================
# FRAUD MART — Test query: fraud rate by transaction type
# Joins fact → dimension (proves the star schema works)
# ============================================================

spark.sql(f"""
          select 
          t.type_name,
          count(*) AS TOTAL_TRANSACTIONS,
          SUM(f.is_fraud) AS fraud_count,
          ROUND(SUM(f.is_fraud) * 100 / COUNT(*), 3) AS fraud_rate_pct
          FROM {GOLD}.fct_transactions f 
          JOIN {GOLD}.dim_transaction_type t 
          on f.type_id = t.type_id
          GROUP BY t.type_name
          ORDER BY fraud_rate_pct DESC
          """).show()